In [ ]:
import json
import glob
from pathlib import Path, WindowsPath
import pathlib

import pandas as pd
import numpy as np

import re

In [ ]:
# Input parameters

pattern = r'mask_tf(\d+)_apoc_cell(\d+)\.json' # RE pattern, in this case to extract apoc-segmented data
exp_folder = r'./' # Folder of the analysis instance
seg_folder_path = r'H:\PROJECTS-03\Pablo\oscillating\a_tw\iu_exp\ppf037\segmentation_ppf037\ppf037' # Root folder of segmentation files
exp_name = 'ppf037' # Experiment name
output_name = 'contour_data_'+ exp_name

In [ ]:
root_folder = Path(seg_folder_path)
output_path = Path(exp_folder).joinpath(Path(output_name))
subdirectories = list(root_folder.glob('*/'))
subdirectories = [subdir for subdir in subdirectories if subdir.is_dir()]
dic = {}

In [ ]:
subdirectories

In [ ]:
output_path

In [ ]:
# Ordering functions
def ordering_function_tf(path):
    """Give the timeframe contained in the name of the json file as an integer.
    Use to sort metadata files"""
    
    pattern = r'mask_tf(\d+)_apoc_cell(\d+)\.json'
    match = re.match(pattern, path.name)
    return int(match.group(1))

def ordering_function_cell(path):
    """Give the cell number contained in the name of the json file as an integer.
    Use to sort metadata files"""

    pattern = r'mask_tf(\d+)_apoc_cell(\d+)\.json'
    match = re.match(pattern, path.name)
    return int(match.group(2))

In [ ]:
def process_subdirectory(subdirectory_path, pattern=pattern, dic=dic):
    json_files = list(subdirectory_path.glob('mask_tf*_apoc_cell*.json'))
    json_files.sort(key=lambda x: (ordering_function_cell(x), ordering_function_tf(x)))
    cells = {re.match(pattern, file.name).group(2) for file in json_files}  # Cell numbers as in the metadata file names

    # For each cell sort by tf and extract the x and y coordinates of the contour
    if cells:
        for cell in cells:
            print(f'Recording contour of cell {subdirectory.name}_{cell}')
            column_name = subdirectory_path.name + '_' + cell
            json_cell = [json_file for json_file in json_files if 'apoc_cell' + cell in json_file.stem]
            tfs = [int((re.match(pattern, json_file.name)).group(1)) for json_file in json_cell]
            tfs.sort()

            for (tf, json_file) in zip(tfs, json_cell):
                with json_file.open('r') as file:
                    data_dict = json.load(file)
                    dic.setdefault(column_name, {}).setdefault(tf, {
                        'x_coords': data_dict.get('xcoords'),
                        'y_coords': data_dict.get('ycoords')
                    })
    else:
        print(f'No cell metadata in {subdirectory_path.name}')

    return dic

In [ ]:
for subdirectory in subdirectories:
    process_subdirectory(subdirectory)

In [ ]:
# Initialize lists to store data for DataFrame
data = []

# Extract data from the nested dictionary
for column_name, time_data in dic.items():
    cell_name = column_name
    
    for tf, coord_data in time_data.items():
        data.append({
            'Cell Name': cell_name,
            'Time Frame': tf,
            'x_coords': coord_data.get('x_coords', []),
            'y_coords': coord_data.get('y_coords', [])
        })

# Create the DataFrame
df = pd.DataFrame(data)

# Optionally, set the time frame as the index
# df.set_index('Time Frame', inplace=True)
print(df)

In [ ]:
df.to_csv(output_path.as_posix())